# Dự đoán lưu lượng giao thông theo thời gian

## Mục tiêu và phạm vi

Dự đoán `traffic_volume` (xe/giờ) từ đặc trưng thời gian. Phần 1 thử **trung bình theo thứ–giờ, Linear Regression, Ridge, Random Forest, Gradient Boosting và MLP**, chưa dùng ngày lễ hoặc thời tiết. Các phần tiếp theo dành cho ngày lễ và so sánh đặc trưng.

| Tập | Vai trò |
|---|---|
| Train | Tính baseline, tạo các fold theo thời gian và fit mô hình |
| Validation | Đánh giá và so sánh các phương án đang phát triển |
| Test | Dành cho đánh giá cuối sau khi chốt quy trình |

**Luồng thực hiện:** chuẩn bị dữ liệu → baseline thứ–giờ → tuning Ridge trong train → đánh giá validation → thử các mô hình khác.

> Các output đã lưu là kết quả của lần chạy trước. Sau khi sửa code hoặc dữ liệu, restart kernel và chạy lại từ đầu để xác nhận tính tái lập.

# Phần 1 · Time-only, chưa dùng ngày lễ

## 1.1 · Chuẩn bị dữ liệu

### Thiết lập thư viện

In [2]:
from pathlib import Path
import hashlib
import sys
import pandas as pd
import numpy as np

### Đọc CSV gốc

Đường dẫn: `data/Metro_Interstate_Traffic_Volume.csv`. Ghi lại môi trường và SHA-256 của dữ liệu.

Kiểm tra tập dữ liệu không rỗng, cột bắt buộc và cấu trúc trước khi xử lý.

In [3]:
# Tìm gốc dự án từ thư mục hiện tại hoặc các thư mục cha.
current_dir = Path.cwd().resolve()
project_dir = next(
    (p for p in (current_dir, *current_dir.parents)
     if (p / "data" / "Metro_Interstate_Traffic_Volume.csv").is_file()),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Không tìm thấy CSV trong thư mục dự án.")

data_path = (
    project_dir
    / "data"
    / "Metro_Interstate_Traffic_Volume.csv"
)

df_raw = pd.read_csv(data_path, keep_default_na=False) ## dữ liệu thô, chưa xử lý missing value, outlier, feature engineering

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)
print("Dataset:", data_path.resolve())
print("SHA-256:", hashlib.sha256(data_path.read_bytes()).hexdigest())
print("Shape:", df_raw.shape)

display(df_raw.head())

Python executable: e:\Project\SGU_HocMay_2026-2027\.venv\Scripts\python.exe
Python version: 3.14.7
Pandas version: 3.0.6
Dataset: E:\Project\SGU_HocMay_2026-2027\metro_interstate_traffic_volume\data\Metro_Interstate_Traffic_Volume.csv
SHA-256: 749c90d720360a4215bb15345526073c079ba4cc95e3fa558796d083f85fce9e
Shape: (48204, 9)


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,None,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,None,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,None,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,None,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,None,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [4]:
required_columns = {"date_time", "traffic_volume"}
missing_columns = required_columns - set(df_raw.columns)

if missing_columns: #Kiểm tra xem có thiếu cột bắt buộc không
    raise ValueError(f"Thiếu cột bắt buộc: {sorted(missing_columns)}")

if df_raw.empty:
    raise ValueError("Dataset không có dòng dữ liệu.")

if not df_raw.columns.is_unique:
    raise ValueError("DataFrame có tên cột bị trùng.")

print("Số dòng:", len(df_raw))
print("Số cột:", len(df_raw.columns))

display(
    pd.DataFrame({
        "dtype": df_raw.dtypes.astype(str),
        "null_count": df_raw.isna().sum(),
    })
)

Số dòng: 48204
Số cột: 9


,dtype,null_count
holiday,str,0
temp,float64,0
rain_1h,float64,0
snow_1h,float64,0
clouds_all,int64,0
weather_main,str,0
weather_description,str,0
date_time,str,0
traffic_volume,int64,0


Cell trên kiểm tra null theo cách đọc CSV hiện tại; chuỗi rỗng và các giá trị bất thường cần được kiểm tra riêng.

### Kiểm tra timestamp và mục tiêu

Chuyển kiểu dữ liệu; dừng nếu timestamp không hợp lệ hoặc target thiếu, không hữu hạn hay âm.

In [5]:
# 1. Kiểm tra hai cột bắt buộc cho baseline Time-only
required_columns = ["date_time", "traffic_volume"]

missing_columns = set(required_columns) - set(df_raw.columns)

if missing_columns:
    raise ValueError(f"Thiếu cột bắt buộc: {sorted(missing_columns)}")

if df_raw.empty:
    raise ValueError("Dataset không có dòng dữ liệu.")

# 2. Tạo bản xử lý riêng, giữ nguyên df_raw
data = df_raw[required_columns].copy()

data["date_time"] = pd.to_datetime(
    data["date_time"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce",
)

data["traffic_volume"] = pd.to_numeric(
    data["traffic_volume"],
    errors="coerce",
)

# 3. Xác định các dòng không hợp lệ
invalid_time = data["date_time"].isna()
invalid_target = ~np.isfinite(data["traffic_volume"])
negative_target = data["traffic_volume"] < 0

print("Timestamp không hợp lệ:", invalid_time.sum())
print("Mục tiêu thiếu hoặc không hữu hạn:", invalid_target.sum())
print("Mục tiêu âm:", negative_target.sum())

invalid_rows = invalid_time | invalid_target | negative_target

if invalid_rows.any():
    display(df_raw.loc[invalid_rows, required_columns].head(10))
    raise ValueError("Có dữ liệu không hợp lệ; cần kiểm tra trước khi tiếp tục.")

print("Đạt kiểm tra kiểu dữ liệu và miền giá trị cơ bản.")
display(data.head())
print(data.dtypes)

Timestamp không hợp lệ: 0
Mục tiêu thiếu hoặc không hữu hạn: 0
Mục tiêu âm: 0
Đạt kiểm tra kiểu dữ liệu và miền giá trị cơ bản.


,date_time,traffic_volume
0,2012-10-02 09:00:00,5545
1,2012-10-02 10:00:00,4516
2,2012-10-02 11:00:00,4767
3,2012-10-02 12:00:00,5026
4,2012-10-02 13:00:00,4918


date_time         datetime64[us]
traffic_volume             int64
dtype: object


### Gộp timestamp trùng

Kiểm tra target trong từng nhóm trước khi giữ một giá trị. Quy tắc này chỉ áp dụng cho `date_time` và `traffic_volume`; chưa xử lý các cột thời tiết.

#### Ví dụ dữ liệu mâu thuẫn giả định

In [6]:
# Ví dụ giả lập: kiểm tra mục tiêu mâu thuẫn tại cùng timestamp
import pandas as pd

example = pd.DataFrame({
    "date_time": [
        "2012-10-10 07:00:00",
        "2012-10-10 07:00:00",
        "2012-10-10 08:00:00",
        "2012-10-10 08:00:00",
        "2012-10-10 09:00:00"
    ],
    "traffic_volume": [
        6793,
        6500,  # Mâu thuẫn với 6793
        6283,
        6283,  # Trùng nhưng không mâu thuẫn
        5680
    ]
})

# Đếm số giá trị mục tiêu khác nhau tại mỗi timestamp
example["target_count"] = (
    example.groupby("date_time")["traffic_volume"]
    .transform("nunique")
)

# Đánh dấu trạng thái
example["status"] = example["target_count"].apply(
    lambda n: "Mâu thuẫn" if n > 1 else "Không mâu thuẫn"
)

display(example)

,date_time,traffic_volume,target_count,status
0,2012-10-10 07:00:00,6793,2,Mâu thuẫn
1,2012-10-10 07:00:00,6500,2,Mâu thuẫn
2,2012-10-10 08:00:00,6283,1,Không mâu thuẫn
3,2012-10-10 08:00:00,6283,1,Không mâu thuẫn
4,2012-10-10 09:00:00,5680,1,Không mâu thuẫn


#### Thực hiện gộp trùng

In [7]:
# Kiểm tra mục tiêu trong từng timestamp trước khi gộp
assert not invalid_rows.any(), "Cần xử lý các dòng không hợp lệ trước khi gộp."

timestamp_summary = data.groupby("date_time").agg(
    row_count=("traffic_volume", "size"),
    target_count=("traffic_volume", "nunique"),
)
conflicting_times = timestamp_summary["target_count"] > 1

print("Timestamp duy nhất:", len(timestamp_summary))
print("Timestamp có nhiều dòng:", (timestamp_summary["row_count"] > 1).sum())
print("Timestamp có mục tiêu mâu thuẫn:", conflicting_times.sum())

if conflicting_times.any():
    # Lấy các timestamp có mục tiêu mâu thuẫn
    conflict_timestamps = timestamp_summary.index[conflicting_times]

    # Lấy các dòng dữ liệu gốc thuộc timestamp đó
    conflict_rows = data[
        data["date_time"].isin(conflict_timestamps)
    ].sort_values("date_time")

    print("Các timestamp có mục tiêu mâu thuẫn:")
    display(timestamp_summary.loc[conflicting_times].head(10))

    print("Chi tiết các dòng mâu thuẫn:")
    display(
        conflict_rows[
            ["date_time", "traffic_volume"]
        ].head(20)
    )

    raise ValueError(
        "Cần kiểm tra mục tiêu mâu thuẫn trước khi gộp."
    )

# Các mục tiêu trong mỗi nhóm đã giống nhau: giữ một giá trị, không cộng
hourly = (
    data.groupby("date_time", as_index=False)
    .agg(traffic_volume=("traffic_volume", "first"))
    .sort_values("date_time")
    .reset_index(drop=True)
)

print("Dòng trước gộp:", len(data))
print("Dòng sau gộp:", len(hourly))
print("Dòng dư đã loại:", len(data) - len(hourly))
print("Timestamp trùng sau gộp:", hourly["date_time"].duplicated().sum())

# Ví dụ nhóm có nhiều dòng và giá trị mục tiêu giữ lại
sample = timestamp_summary[timestamp_summary["row_count"] > 1].head(5)
display(sample.join(hourly.set_index("date_time")))


Timestamp duy nhất: 40575
Timestamp có nhiều dòng: 5445
Timestamp có mục tiêu mâu thuẫn: 0
Dòng trước gộp: 48204
Dòng sau gộp: 40575
Dòng dư đã loại: 7629
Timestamp trùng sau gộp: 0


,row_count,target_count,traffic_volume
date_time,,,
2012-10-10 07:00:00,2,1,6793
2012-10-10 08:00:00,2,1,6283
2012-10-10 09:00:00,2,1,5680
2012-10-14 09:00:00,3,1,2685
2012-10-14 10:00:00,2,1,3370


### Kiểm tra độ liên tục và chia theo thời gian

Dữ liệu gồm các giờ đã quan sát; không nội suy các giờ thiếu. Tỷ lệ 70/15/15 tính theo số timestamp quan sát được.

### Xét tính liên tục thời gian của bộ dữ liệu, kiểm tra đứt gãy chuỗi thời gian

In [8]:
times = hourly["date_time"]

assert times.notna().all(), "Timestamp có giá trị thiếu."
assert times.is_unique, "Timestamp vẫn còn trùng."
assert times.is_monotonic_increasing, "Thời gian chưa được sắp xếp."
assert times.eq(times.dt.floor("h")).all(), "Timestamp không nằm đúng mốc giờ."

# Khoảng cách giữa hai timestamp liên tiếp
time_diff = times.diff()
gap_mask = time_diff > pd.Timedelta(hours=1)

# Các khoảng bị thiếu giờ
gaps = pd.DataFrame({
    "Trước khoảng trống": times.shift(1)[gap_mask],
    "Sau khoảng trống": times[gap_mask],
})

gaps["Số giờ thiếu"] = (
    (
        gaps["Sau khoảng trống"]
        - gaps["Trước khoảng trống"]
    ) / pd.Timedelta(hours=1) - 1
).astype(int)

print("Timestamp quan sát được:", len(times))
print("Số khoảng đứt gãy:", len(gaps))
print("Tổng số giờ thiếu:", gaps["Số giờ thiếu"].sum())

display(
    gaps.sort_values("Số giờ thiếu", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

Timestamp quan sát được: 40575
Số khoảng đứt gãy: 2588
Tổng số giờ thiếu: 11976


,Trước khoảng trống,Sau khoảng trống,Số giờ thiếu
0,2014-08-08 01:00:00,2015-06-11 20:00:00,7386
1,2013-10-27 01:00:00,2013-11-06 04:00:00,242
2,2015-06-14 20:00:00,2015-06-19 18:00:00,117
3,2014-04-29 08:00:00,2014-05-04 05:00:00,116
4,2015-10-23 11:00:00,2015-10-27 08:00:00,92
5,2014-07-27 16:00:00,2014-07-30 09:00:00,64
6,2013-08-31 00:00:00,2013-09-01 23:00:00,46
7,2015-06-22 18:00:00,2015-06-24 04:00:00,33
8,2014-08-02 02:00:00,2014-08-03 11:00:00,32
9,2013-09-03 02:00:00,2013-09-04 08:00:00,29


### Thực hiện chia train/validation/test

In [9]:

# Chia theo vị trí trên dữ liệu đã sắp xếp thời gian
n_rows = len(hourly)
train_end = int(n_rows * 0.70)
validation_end = int(n_rows * 0.85)
assert 0 < train_end < validation_end < n_rows, "Không đủ dữ liệu để chia ba tập."

train = hourly.iloc[:train_end].copy()
validation = hourly.iloc[train_end:validation_end].copy()
test = hourly.iloc[validation_end:].copy()

split_summary = pd.DataFrame([
    {
        "Tập": name,
        "Số dòng": len(subset),
        "Tỷ lệ (%)": round(100 * len(subset) / n_rows, 2),
        "Bắt đầu": subset["date_time"].min(),
        "Kết thúc": subset["date_time"].max(),
    }
    for name, subset in [
        ("Train", train),
        ("Validation", validation),
        ("Test", test),
    ]
])
display(split_summary)


,Tập,Số dòng,Tỷ lệ (%),Bắt đầu,Kết thúc
0,Train,28402,70.0,2012-10-02 09:00:00,2017-05-10 04:00:00
1,Validation,6086,15.0,2017-05-10 05:00:00,2018-01-19 14:00:00
2,Test,6087,15.0,2018-01-19 15:00:00,2018-09-30 23:00:00


### Tạo đặc trưng thời gian

Tách `weekday` và `hour` cho train và validation. Các đặc trưng này dùng để dự đoán; việc tạo chúng chưa đo được ảnh hưởng riêng của từng đặc trưng.

In [10]:
# Tạo bản sao để giữ nguyên các tập sau khi chia
train_time = train.copy()
validation_time = validation.copy()

# Tách thứ trong tuần và giờ từ timestamp
train_time["weekday"] = train_time["date_time"].dt.dayofweek
train_time["hour"] = train_time["date_time"].dt.hour

validation_time["weekday"] = validation_time["date_time"].dt.dayofweek
validation_time["hour"] = validation_time["date_time"].dt.hour

display(
    train_time[["date_time", "weekday", "hour", "traffic_volume"]].head()
)

,date_time,weekday,hour,traffic_volume
0,2012-10-02 09:00:00,1,9,5545
1,2012-10-02 10:00:00,1,10,4516
2,2012-10-02 11:00:00,1,11,4767
3,2012-10-02 12:00:00,1,12,5026
4,2012-10-02 13:00:00,1,13,4918


## 1.2 · Baseline trung bình theo thứ–giờ

Mỗi dự đoán là trung bình target của cùng cặp `(weekday, hour)` trong train. Nếu cặp chưa xuất hiện, dùng trung bình toàn train.

### Kiểm tra số mẫu theo từng cặp thứ-giờ

In [11]:
weekday_hour_stats = (
    train_time
    .groupby(["weekday", "hour"])["traffic_volume"]
    .agg(count="count", mean="mean")
)

print("Số nhóm thứ–giờ có dữ liệu:", len(weekday_hour_stats))
print("Số mẫu ít nhất trong một nhóm:", weekday_hour_stats["count"].min())
print("Số mẫu nhiều nhất trong một nhóm:", weekday_hour_stats["count"].max())

display(weekday_hour_stats.head(24)) ## ngày thứ 2

Số nhóm thứ–giờ có dữ liệu: 168
Số mẫu ít nhất trong một nhóm: 155
Số mẫu nhiều nhất trong một nhóm: 183


count         mean
weekday hour                    
0       0       177   617.389831
        1       168   376.029762
        2       174   287.919540
        3       168   335.505952
        4       170   792.541176
        5       175  2549.611429
        6       171  5168.578947
        7       173  5869.069364
        8       165  5252.466667
        9       164  4569.609756
        10      169  4110.236686
        11      167  4324.071856
        12      169  4569.201183
        13      167  4609.323353
        14      166  4925.885542
        15      168  5339.821429
        16      166  5950.156627
        17      166  5623.891566
        18      168  4136.541667
        19      173  3012.254335
        20      168  2552.541667
        21      177  2254.435028
        22      171  1673.216374
        23      173  1098.294798

### Kiểm tra các nhóm có ít mẫu nhất

In [12]:
display(
    weekday_hour_stats
    .sort_values("count")
    .head(10)
)

count         mean
weekday hour                    
2       9       155  5029.929032
1       7       156  6150.032051
2       17      159  6048.729560
        7       160  6173.375000
        13      160  4879.687500
1       5       160  2725.468750
5       6       160  1297.043750
1       9       161  4986.093168
        17      161  5975.981366
3       9       161  5021.012422

### Tạo bảng tra cứu từ train

In [13]:
# Lưu lượng trung bình của từng cặp thứ–giờ trong train
weekday_hour_mean = weekday_hour_stats["mean"]

# Trung bình toàn train: dùng dự phòng nếu không tìm được cặp thứ–giờ
train_global_mean = train_time["traffic_volume"].mean()

display(weekday_hour_mean.head())
print("Trung bình toàn train:", train_global_mean)

weekday  hour
0        0       617.389831
         1       376.029762
         2       287.919540
         3       335.505952
         4       792.541176
Name: mean, dtype: float64

Trung bình toàn train: 3264.604218012816


### Tra cứu nhóm của validation

In [14]:
# Tạo khóa (weekday, hour) cho từng dòng validation
validation_keys = pd.MultiIndex.from_frame(
    validation_time[["weekday", "hour"]]
)

# Tra trung bình train theo đúng thứ tự các dòng validation
validation_lookup = weekday_hour_mean.reindex(validation_keys)

missing_groups = validation_lookup.isna().sum()

print("Số dòng validation:", len(validation_time))
print("Số dòng không tìm thấy nhóm trong train:", missing_groups)

display(validation_lookup.head())

Số dòng validation: 6086
Số dòng không tìm thấy nhóm trong train: 0


weekday  hour
2        5       2752.170732
         6       5561.771930
         7       6173.375000
         8       5754.736842
         9       5029.929032
Name: mean, dtype: float64

### Tạo dự đoán và kiểm tra kết quả

In [15]:
baseline_predictions = (
    validation_lookup
    .fillna(train_global_mean)
    .to_numpy()
)

validation_results = validation_time[
    ["date_time", "weekday", "hour", "traffic_volume"]
].copy()

validation_results["prediction"] = baseline_predictions

display(validation_results.head(10))

,date_time,weekday,hour,traffic_volume,prediction
28402,2017-05-10 05:00:00,2,5,2920,2752.170732
28403,2017-05-10 06:00:00,2,6,5988,5561.771930
28404,2017-05-10 07:00:00,2,7,6739,6173.375000
28405,2017-05-10 08:00:00,2,8,5847,5754.736842
28406,2017-05-10 09:00:00,2,9,5062,5029.929032
28407,2017-05-10 10:00:00,2,10,4596,4469.971751
28408,2017-05-10 11:00:00,2,11,4983,4668.317073
28409,2017-05-10 12:00:00,2,12,4938,4858.810651
28410,2017-05-10 13:00:00,2,13,5026,4879.687500
28411,2017-05-10 14:00:00,2,14,5050,5219.011429


### Đánh giá trên validation

MAE và RMSE đo sai số dự đoán; R² dùng trung bình target của tập đánh giá làm mốc trong công thức.

In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_validation = validation_results["traffic_volume"]
baseline_prediction = validation_results["prediction"]

baseline_metrics = {
    "MAE": mean_absolute_error(y_validation, baseline_prediction),
    "RMSE": np.sqrt(mean_squared_error(y_validation, baseline_prediction)),
    "R2": r2_score(y_validation, baseline_prediction),
}

print("Số mẫu validation:", len(y_validation))
print("Số dòng dùng fallback:", missing_groups)
print("Kết quả baseline:", baseline_metrics)

Số mẫu validation: 6086
Số dòng dùng fallback: 0
Kết quả baseline: {'MAE': 285.0272011750431, 'RMSE': np.float64(501.40716261706484), 'R2': 0.9349371593454914}


### Nhận xét từ output đã lưu

- Train có đủ 168 cặp thứ–giờ; số mẫu mỗi nhóm từ 155 đến 183. Validation có 6.086 giờ và không có dòng dùng fallback.
- MAE: `285.0272011750431` xe/giờ; RMSE: `501.40716261706484` xe/giờ; R²: `0.9349371593454914` trên validation.
- MAE là sai số tuyệt đối **trung bình**, không phải sai số của từng dự đoán. Metrics chỉ mô tả các giờ có quan sát trong validation; dữ liệu còn thiếu nhiều giờ và mô hình chưa dùng ngày lễ.

## 1.3 · Huấn luyện và tuning mô hình

Giữ cùng đặc trưng và tập validation ban đầu. Tuning dùng các fold theo thứ tự thời gian trong train; test dành cho đánh giá cuối.

### Chuẩn bị X và y

`X` gồm thứ và giờ; `y` là lưu lượng. Giữ cùng thứ tự dòng như `train_time` để kiểm tra mốc fold.

In [17]:
# Các thông tin lịch sử dụng cho lần chạy đầu tiên
time_features = ["weekday", "hour"]

X_train = train_time[time_features]
y_train = train_time["traffic_volume"]

X_validation = validation_time[time_features]
y_validation = validation_time["traffic_volume"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

display(X_train.head())

X_train: (28402, 2)
y_train: (28402,)
X_validation: (6086, 2)
y_validation: (6086,)


,weekday,hour
0,1,9
1,1,10
2,1,11
3,1,12
4,1,13


### Kiểm tra mốc các fold

Các fold chia theo số dòng quan sát, không phải theo năm lịch; khoảng thời gian đánh giá có thể chứa các đoạn thiếu giờ.

In [18]:
from sklearn.model_selection import TimeSeriesSplit

time_cv = TimeSeriesSplit(n_splits=3)

fold_rows = []

for fold, (train_idx, val_idx) in enumerate(time_cv.split(X_train), start=1):
    fold_train = train_time.iloc[train_idx]
    fold_val = train_time.iloc[val_idx]

    fold_rows.append({
        "fold": fold,
        "train_start": fold_train["date_time"].min(),
        "train_end": fold_train["date_time"].max(),
        "train_samples": len(fold_train),
        "val_start": fold_val["date_time"].min(),
        "val_end": fold_val["date_time"].max(),
        "val_samples": len(fold_val),
    })

display(pd.DataFrame(fold_rows))

,fold,train_start,train_end,train_samples,val_start,val_end,val_samples
0,1,2012-10-02 09:00:00,2013-08-18 18:00:00,7102,2013-08-18 19:00:00,2015-07-07 11:00:00,7100
1,2,2012-10-02 09:00:00,2015-07-07 11:00:00,14202,2015-07-07 12:00:00,2016-07-15 04:00:00,7100
2,3,2012-10-02 09:00:00,2016-07-15 04:00:00,21302,2016-07-15 05:00:00,2017-05-10 04:00:00,7100


### a. Linear Regression & Ridge
Linear Regression tạo dự đoán từ tổng có trọng số của các đặc trưng:

$$\hat y = b + w_1x_1 + w_2x_2 + \dots + w_px_p$$

Trong đó x là đầu vào, w là hệ số được học từ train, b là hệ số chặn và $\hat y$ là dự đoán. Mô hình tìm hệ số để giảm tổng bình phương sai số trên train.

Ridge là Linear Regression có thêm cơ chế phạt hệ số quá lớn:
$$
\text{Mục tiêu Ridge}
=
\underbrace{\sum(y_i-\hat y_i)^2}_{\text{sai số trên train}}
+
\underbrace{\alpha\sum w_j^2}_{\text{phạt hệ số}}
$$
$\alpha$ là siêu tham số điều khiển độ mạnh của phần phạt. alpha lớn hơn thường làm hệ số bị kéo về gần 0 mạnh hơn; giá trị quá lớn có thể khiến mô hình quá đơn giản

#### Tuning Ridge trong train

Encoder được fit trong từng fold. Chọn `alpha` theo MAE trung bình CV; sau đó fit lại cấu hình được chọn trên toàn train.

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("calendar", OneHotEncoder(handle_unknown="ignore"), time_features)
    ]
)

ridge_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", Ridge()),
    ]
)
 
# Giữ thứ tự thời gian của các dòng train.
time_cv = TimeSeriesSplit(n_splits=3)

ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid={"model__alpha": [0.1, 1.0, 10.0, 100.0, 1000.0]},
    scoring="neg_mean_absolute_error",
    cv=time_cv,
    refit=True,
    return_train_score=True,
    error_score="raise",
    n_jobs=1,
)

ridge_search.fit(X_train, y_train)

print("Alpha được chọn:", ridge_search.best_params_["model__alpha"])
print("MAE trung bình CV:", -ridge_search.best_score_)

Alpha được chọn: 0.1
MAE trung bình CV: 594.3735458295135


#### Kết quả tuning theo tham số

In [20]:
ridge_cv_results = pd.DataFrame({
    "alpha": ridge_search.cv_results_["param_model__alpha"],
    "mean_train_mae": -ridge_search.cv_results_["mean_train_score"],
    "mean_cv_mae": -ridge_search.cv_results_["mean_test_score"],
    "std_cv_mae": ridge_search.cv_results_["std_test_score"],
    "rank": ridge_search.cv_results_["rank_test_score"],
}).sort_values("rank")

display(ridge_cv_results)

,alpha,mean_train_mae,mean_cv_mae,std_cv_mae,rank
0,0.1,595.022506,594.373546,10.639483,1
1,1.0,595.692791,594.884476,10.769361,2
2,10.0,602.764602,600.377899,12.686170,3
3,100.0,690.936803,676.808974,55.853945,4
4,1000.0,1237.138390,1200.304894,157.318033,5


#### Đánh giá Ridge trên validation

Baseline dùng trung bình riêng cho từng cặp thứ–giờ; Ridge hiện dùng one-hot riêng cho thứ và giờ, chưa có tương tác của từng cặp.

In [21]:
ridge_prediction = ridge_search.predict(X_validation)

ridge_metrics = {
    "MAE": mean_absolute_error(y_validation, ridge_prediction),
    "RMSE": np.sqrt(mean_squared_error(y_validation, ridge_prediction)),
    "R2": r2_score(y_validation, ridge_prediction),
}

display(pd.DataFrame(
    [baseline_metrics, ridge_metrics],
    index=["Baseline thứ–giờ", "Ridge sau tuning"],
))

,MAE,RMSE,R2
Baseline thứ–giờ,285.027201,501.407163,0.934937
Ridge sau tuning,581.941398,807.954197,0.831063


### b. Linear Regression — mốc tuyến tính không phạt

One-hot mã hóa thứ và giờ; Linear Regression học tổng các hệ số sao cho sai số bình phương trên train nhỏ nhất. Cấu hình này không tự tạo tương tác giữa từng cặp thứ–giờ.

| Thành phần | Thiết lập | Ý nghĩa |
|---|---|---|
| Đặc trưng | `weekday`, `hour` | Cùng thông tin gốc với baseline |
| Mã hóa | One-hot | Mỗi giá trị có một cột chỉ báo |
| Mô hình | Linear Regression | Không có tham số phạt để tuning |

In [22]:
from sklearn.linear_model import LinearRegression

linear_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])
linear_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](2,)","['weekday','hour']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,2
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('calendar', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For datafra

In [23]:
linear_prediction = linear_pipeline.predict(X_validation)
linear_metrics = {
    "MAE": mean_absolute_error(y_validation, linear_prediction),
    "RMSE": np.sqrt(mean_squared_error(y_validation, linear_prediction)),
    "R2": r2_score(y_validation, linear_prediction),
}
display(pd.DataFrame([linear_metrics], index=["Linear Regression"]))

,MAE,RMSE,R2
Linear Regression,581.907324,807.95664,0.831062


### c. Công cụ dùng chung cho tuning

Với mỗi bộ tham số, `GridSearchCV` học trên quá khứ và đo MAE ở phần thời gian kế tiếp của từng fold. Tham số có MAE trung bình nhỏ nhất được fit lại trên toàn train; validation chỉ dùng đánh giá sau đó. `error_score="raise"` giúp lỗi không bị che giấu.

In [37]:
def tune_model(estimator, parameters):
    """Chọn tham số theo MAE trên ba fold theo thứ tự thời gian."""
    search = GridSearchCV(
        estimator=estimator,
        param_grid=parameters,
        scoring="neg_mean_absolute_error",
        cv=time_cv,
        refit=True,
        return_train_score=True,
        error_score="raise",
        n_jobs=1,
    )
    return search.fit(X_train, y_train)

In [38]:
def summarize_search(search):
    """Trả bảng tham số và MAE của từng fold, giữ nguyên độ chính xác."""
    rows = pd.DataFrame(search.cv_results_)
    result = pd.DataFrame({
        "parameters": rows["params"],
        "fold_1_mae": -rows["split0_test_score"],
        "fold_2_mae": -rows["split1_test_score"],
        "fold_3_mae": -rows["split2_test_score"],
        "mean_cv_mae": -rows["mean_test_score"],
        "rank": rows["rank_test_score"],
    })
    return result.sort_values("rank").reset_index(drop=True)

def validation_metrics(prediction):
    """Đánh giá dự đoán trên cùng các giờ validation."""
    return {
        "MAE": mean_absolute_error(y_validation, prediction),
        "RMSE": np.sqrt(mean_squared_error(y_validation, prediction)),
        "R2": r2_score(y_validation, prediction),
    }

### d. Random Forest

Mỗi cây học các phép chia; dự đoán của rừng là trung bình dự đoán của các cây. Cây có thể học tương tác thứ–giờ. Đầu vào được giữ dạng số nguyên; cây chia theo ngưỡng, nên cách biểu diễn khác Ridge one-hot.

| Tham số | Giá trị thử | Ý nghĩa |
|---|---|---|
| `n_estimators` | 40 | Số cây, cố định để thử nhanh |
| `max_depth` | 8, 16 | Giới hạn độ sâu cây |
| `min_samples_leaf` | 5, 20 | Số mẫu tối thiểu tại lá |
| `random_state` | 42 | Tái lập quá trình lấy mẫu |

In [39]:
from sklearn.ensemble import RandomForestRegressor

forest = RandomForestRegressor(n_estimators=40, random_state=42, n_jobs=1)
forest_grid = {
    "max_depth": [8, 16],
    "min_samples_leaf": [5, 20],
}
forest_search = tune_model(forest, forest_grid)

In [40]:
forest_cv_results = summarize_search(forest_search)
forest_metrics = validation_metrics(forest_search.predict(X_validation))
display(forest_cv_results)
print("Cấu hình chọn:", forest_search.best_params_)
print("Validation:", forest_metrics)

,parameters,fold_1_mae,fold_2_mae,fold_3_mae,mean_cv_mae,rank
0,"{'max_depth': 16, 'min_samples_leaf': 20}",316.991575,286.512161,306.795615,303.433117,1
1,"{'max_depth': 16, 'min_samples_leaf': 5}",317.047365,286.512161,306.795615,303.451714,2
2,"{'max_depth': 8, 'min_samples_leaf': 20}",319.631421,290.758344,309.738874,306.709546,3
3,"{'max_depth': 8, 'min_samples_leaf': 5}",319.706162,290.758344,309.738874,306.734460,4


Cấu hình chọn: {'max_depth': 16, 'min_samples_leaf': 20}
Validation: {'MAE': 284.84865385759554, 'RMSE': np.float64(501.57788653880664), 'R2': 0.9348928453620192}


### e. Gradient Boosting

Các cây được thêm tuần tự; mỗi cây mới học theo hướng giảm sai số của tổ hợp hiện tại. Với mất mát bình phương, trực giác là học phần dư. `learning_rate` điều khiển mức đóng góp của mỗi cây.

| Tham số | Giá trị thử | Ý nghĩa |
|---|---|---|
| `n_estimators` | 80 | Số cây thêm tuần tự |
| `learning_rate` | 0.05, 0.1 | Trọng số đóng góp của cây mới |
| `max_depth` | 2, 3 | Độ sâu mỗi cây |
| `random_state` | 42 | Tái lập quá trình chạy |

In [41]:
from sklearn.ensemble import GradientBoostingRegressor

boost = GradientBoostingRegressor(n_estimators=80, random_state=42)
boost_grid = {
    "learning_rate": [0.05, 0.1],
    "max_depth": [2, 3],
}
boost_search = tune_model(boost, boost_grid)

In [42]:
boost_cv_results = summarize_search(boost_search)
boost_metrics = validation_metrics(boost_search.predict(X_validation))
display(boost_cv_results)
print("Cấu hình chọn:", boost_search.best_params_)
print("Validation:", boost_metrics)

,parameters,fold_1_mae,fold_2_mae,fold_3_mae,mean_cv_mae,rank
0,"{'learning_rate': 0.1, 'max_depth': 3}",367.742288,339.724945,363.388760,356.951998,1
1,"{'learning_rate': 0.05, 'max_depth': 3}",444.988482,416.898544,441.190315,434.359113,2
2,"{'learning_rate': 0.1, 'max_depth': 2}",512.517331,477.362377,502.996493,497.625400,3
3,"{'learning_rate': 0.05, 'max_depth': 2}",579.059310,559.019229,561.189176,566.422572,4


Cấu hình chọn: {'learning_rate': 0.1, 'max_depth': 3}
Validation: {'MAE': 351.12111870841306, 'RMSE': np.float64(557.0385479851142), 'R2': 0.9196987226227398}


### f. MLP — mạng neuron nhiều lớp

Đầu vào one-hot đi qua lớp ẩn, tạo dự đoán; thuật toán lan truyền gradient để điều chỉnh trọng số. Mục tiêu được chuẩn hóa bằng thống kê train của từng fold rồi đổi dự đoán về xe/giờ. Không dùng early stopping ngẫu nhiên; xem cảnh báo hội tụ sau khi chạy.

| Tham số | Giá trị thử | Ý nghĩa |
|---|---|---|
| `hidden_layer_sizes` | (16,), (32,) | Một lớp ẩn với 16 hoặc 32 neuron |
| `alpha` | 0.0001 | Phạt hệ số L2, cố định |
| `max_iter` | 120 | Tối đa số epoch |
| `random_state` | 42 | Tái lập khởi tạo và thứ tự mẫu |

In [43]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

mlp_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", MLPRegressor(alpha=0.0001, max_iter=120, random_state=42)),
])
mlp = TransformedTargetRegressor(regressor=mlp_pipeline, transformer=StandardScaler())
mlp_grid = {"regressor__model__hidden_layer_sizes": [(16,), (32,)]}
mlp_search = tune_model(mlp, mlp_grid)

In [44]:
mlp_cv_results = summarize_search(mlp_search)
mlp_metrics = validation_metrics(mlp_search.predict(X_validation))
display(mlp_cv_results)
print("Cấu hình chọn:", mlp_search.best_params_)
print("Số epoch:", mlp_search.best_estimator_.regressor_.named_steps["model"].n_iter_)
print("Validation:", mlp_metrics)

,parameters,fold_1_mae,fold_2_mae,fold_3_mae,mean_cv_mae,rank
0,"{'regressor__model__hidden_layer_sizes': (16,)}",321.487005,286.609255,315.634208,307.910156,1
1,"{'regressor__model__hidden_layer_sizes': (32,)}",315.342687,295.671213,312.889895,307.967932,2


Cấu hình chọn: {'regressor__model__hidden_layer_sizes': (16,)}
Số epoch: 25
Validation: {'MAE': 283.3201414126816, 'RMSE': np.float64(504.034571152169), 'R2': 0.9342535051743674}


### g. So sánh validation

Các phương pháp dùng cùng 6.086 giờ validation. Nhóm trung bình là mốc theo cặp thứ–giờ; Ridge/Linear dùng one-hot riêng, cây dùng số nguyên. Chênh lệch phản ánh cả mô hình lẫn cách biểu diễn. Không chọn tham số bằng test.

In [45]:
comparison_all = pd.DataFrame({
    "Trung bình thứ–giờ": baseline_metrics,
    "Linear Regression": linear_metrics,
    "Ridge": ridge_metrics,
    "Random Forest": forest_metrics,
    "Gradient Boosting": boost_metrics,
    "MLP": mlp_metrics,
}).T.sort_values("MAE")
display(comparison_all)

,MAE,RMSE,R2
MLP,283.320141,504.034571,0.934254
Random Forest,284.848654,501.577887,0.934893
Trung bình thứ–giờ,285.027201,501.407163,0.934937
Gradient Boosting,351.121119,557.038548,0.919699
Linear Regression,581.907324,807.956640,0.831062
Ridge,581.941398,807.954197,0.831063


### h. Ghi nhật ký thực nghiệm

Các cell sau chuẩn bị kết quả và gọi trực tiếp `src/experiment_logger.py`. Chạy toàn bộ phần mô hình trước để các kết quả, tham số và fold tương ứng với cùng một lần chạy.

**Chủ động ghi:** đổi `SAVE_EXPERIMENT_LOGS` thành `True` ở cell cuối rồi chạy cell đó. Mỗi lần chạy với `True` tạo sáu file `.txt` mới, không ghi đè. Chuyển lại `False` trước khi chạy lại toàn notebook nếu không muốn tạo bản trùng. Excel không tự cập nhật khi ghi log.

In [50]:
# Cho phép import src khi chạy từ notebooks/experiments/.
import sys
import json
import sklearn

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from src.experiment_logger import save_experiment_log

In [51]:
# Chuẩn bị thông tin chung của lần chạy hiện tại.
dataset_hash = hashlib.sha256(data_path.read_bytes()).hexdigest()
common_observations = [
    f"CSV SHA-256: {dataset_hash}",
    f"Train: {len(train)} giờ; {train.date_time.min()} đến {train.date_time.max()}.",
    f"Validation: {len(validation)} giờ; {validation.date_time.min()} đến {validation.date_time.max()}.",
    f"Test: {len(test)} giờ; giữ riêng, chưa đánh giá.",
    f"Python {sys.version.split()[0]}; pandas {pd.__version__}; scikit-learn {sklearn.__version__}.",
    "Notebook: notebooks/experiments/V_1.ipynb; commit chưa xác minh.",
    "Time-only: chưa dùng ngày lễ, thời tiết hay lịch sử lưu lượng.",
]
common_problems = [
    "Dữ liệu thiếu giờ; validation chỉ gồm các giờ quan sát được.",
    "Các mô hình dùng cách mã hóa khác nhau; kết quả chưa đánh giá test.",
]

In [52]:
# Khai báo rõ kết quả và cấu hình thực tế của từng phương pháp.
experiment_runs = [
    ("Trung bình thứ–giờ", baseline_metrics, None,
     f"Trung bình theo cặp thứ–giờ; số dòng fallback: {missing_groups}."),
    ("Linear Regression", linear_metrics, None,
     "One-hot thứ và giờ; LinearRegression mặc định."),
    ("Ridge", ridge_metrics, ridge_search,
     "One-hot thứ và giờ; alpha thử [0.1, 1, 10, 100, 1000]."),
    ("Random Forest", forest_metrics, forest_search,
     "Thứ và giờ dạng số; n_estimators=40; random_state=42."),
    ("Gradient Boosting", boost_metrics, boost_search,
     "Thứ và giờ dạng số; n_estimators=80; random_state=42."),
    ("MLP", mlp_metrics, mlp_search,
     "One-hot; chuẩn hóa target trong fold; alpha=0.0001; max_iter=120; random_state=42."),
]

In [53]:
# Đổi thành True để chủ động tạo một bộ log mới.
SAVE_EXPERIMENT_LOGS = True

if SAVE_EXPERIMENT_LOGS:
    for name, metrics, search, settings in experiment_runs:
        observations = common_observations + [f"Thiết lập: {settings}"]
        if search is not None:
            cv_details = summarize_search(search).to_json(orient="records")
            observations += [
                "Tuning: 3 fold theo thời gian trong train; chọn theo MAE.",
                f"Tham số được chọn: {search.best_params_}",
                f"Kết quả từng fold: {cv_details}",
                f"Mốc các fold: {fold_rows}",
            ]
        log_path = save_experiment_log(
            model=f"Thử nghiệm baseline - {name}",
            features=time_features,
            results=metrics,
            observations=observations,
            problems=common_problems,
            next_steps=["Phân tích sai số theo giờ và giai đoạn thời gian."],
        )
        print(f"Đã lưu {name}: {log_path}")
else:
    print("Chưa tạo log. Đổi SAVE_EXPERIMENT_LOGS thành True khi cần ghi.")

Đã lưu Trung bình thứ–giờ: E:\Project\SGU_HocMay_2026-2027\metro_interstate_traffic_volume\experiments\EXP_001.txt
Đã lưu Linear Regression: E:\Project\SGU_HocMay_2026-2027\metro_interstate_traffic_volume\experiments\EXP_002.txt
Đã lưu Ridge: E:\Project\SGU_HocMay_2026-2027\metro_interstate_traffic_volume\experiments\EXP_003.txt
Đã lưu Random Forest: E:\Project\SGU_HocMay_2026-2027\metro_interstate_traffic_volume\experiments\EXP_004.txt
Đã lưu Gradient Boosting: E:\Project\SGU_HocMay_2026-2027\metro_interstate_traffic_volume\experiments\EXP_005.txt
Đã lưu MLP: E:\Project\SGU_HocMay_2026-2027\metro_interstate_traffic_volume\experiments\EXP_006.txt


# Phần 2 · Thời gian và ngày lễ

**Chưa triển khai.** Cần xác định nguồn lịch lễ và cách tạo nhãn theo ngày trước khi so sánh với Phần 1.

# Phần 3 · Tổng kết và so sánh

So sánh trên cùng tập giờ và cùng cách đánh giá; ghi rõ đặc trưng, cấu hình, số mẫu và giới hạn. Chưa có kết luận khi chưa chạy thí nghiệm tương ứng.